# Day 2 · Morning exercises (Exercise A)

**09:45–11:10 · then everyone back in the room at 11:10 for Peer review A**

Bayesian statistics for PhD researchers, Aalborg University 2026. Work in your group and keep your own copy of this notebook. Select **R (bayes)** and run the cells in order from a fresh kernel. The code uses R with the tidyverse (dplyr for summaries, ggplot2 for figures); there is no Stan in today's exercises. The afternoon continues in `day-2-exercises-afternoon.ipynb`.

The data are **synthetic teaching examples**.

## Plan for this session

| Part | What | Guide time |
|---|---|---|
| A1 | One posterior by hand | 20 min |
| A2 | A different prior | 25 min |
| A3 | Check by grid | 20 min |
| — | Exit response and your group's question set | 15 min |

**Roles this block:** coder, checker, two question writers, interviewer, reporter. Rotate them in the afternoon.

Take a break when it suits your group. Optional work is not needed for the afternoon.

In [ ]:
options(tidyverse.quiet = TRUE)
library(tidyverse)   # dplyr for summaries, ggplot2 for figures
set.seed(2026)
options(width = 95, repr.plot.width = 8, repr.plot.height = 5, repr.plot.bg = "white")

## A1 (20 min): one posterior by hand

**The setting.** A *trial* is a yes/no experiment, for example one bus departure (on time or not) or one seed (germinates or not). We ran **n = 10 trials** and saw **y = 7 successes**. We want to learn **p, the probability of success in one trial**. (The slides call it θ.)

**The model** (*slides "The likelihood: binomial in y, a curve in θ" and "The beta distribution: a prior for a proportion"*):

- Likelihood: y | p ~ Binomial(10, p), so p(y | p) ∝ p⁷ (1 − p)³.
- Prior: p ~ Beta(1, 1), the flat prior. Its density is ∝ p^(1−1) (1 − p)^(1−1) = 1.

**Tasks.**

1. *By hand:* multiply the likelihood and the prior, combine the powers of p and of 1 − p, and recognise the result as a Beta(?, ?) density (*slide "The beta–binomial update"*).
2. *In R:* compute the posterior mean a / (a + b), the 90% interval `qbeta(c(0.05, 0.95), a, b)` and P(p > 0.5 | y) = `1 - pbeta(0.5, a, b)`, using the posterior a and b (*slide "Summarising the posterior Beta(16, 8)"*).
3. *In words:* interpret each of the three numbers in one sentence.

In [ ]:
n <- 10; y <- 7     # data: 7 successes in 10 trials
a <- 1; b <- 1      # prior Beta(1, 1)

# a_post <- ...
# b_post <- ...
# then: a_post / (a_post + b_post), qbeta(c(0.05, 0.95), a_post, b_post), 1 - pbeta(0.5, a_post, b_post)

**Your derivation and interpretation:** write them here.

## A2 (25 min): a different prior

**The setting.** The same data, 7 successes in 10 trials, but a second person starts from the prior **Beta(5, 5)**. Its mean is also 0.5, but it is more concentrated: you can read it as about 5 + 5 = 10 earlier trials, half of them successes (*slide "How much does the prior matter?"*).

**What a prior expects to see.** A prior for p also says which data we expect *before* any data are collected. We can see this by simulating in two steps: first draw p from the prior, then draw the number of successes in 10 trials for that p:

`rbinom(5000, size = 10, prob = rbeta(5000, a, b))`

This gives 5,000 simulated ten-trial counts. (The afternoon uses the same two steps for prediction.)

**Tasks.**

1. Compute the posterior for the Beta(5, 5) prior, its mean and 90% interval. Compare them with A1.
2. Plot four curves in one figure: both priors and both posteriors. The code cell shows the pattern.
3. Simulate 5,000 ten-trial counts from each prior and draw a bar chart. Which counts does each prior expect?
4. Which prior moves the answer more? What earlier information would justify using Beta(5, 5)?

In [ ]:
# Task 1: the posterior for the Beta(5, 5) prior
# a_post2 <- 5 + y;  b_post2 <- 5 + n - y

# Task 2: one small table per curve, stacked with bind_rows(), then one ggplot()
p_grid <- seq(0.001, 0.999, length.out = 1000)
# curves <- bind_rows(
#   tibble(p = p_grid, density = dbeta(p_grid, 1, 1), curve = "prior Beta(1,1)"),
#   tibble(p = p_grid, density = dbeta(p_grid, ?, ?), curve = "posterior ..."),
#   ...
# )
# ggplot(curves, aes(x = p, y = density, colour = curve)) + geom_line()

# Task 3: counts expected by each prior
# counts_flat <- rbinom(5000, size = 10, prob = rbeta(5000, 1, 1))
# tibble(successes = counts_flat) |> ggplot(aes(x = successes)) + geom_bar()

**Your answer:** write a few sentences here.

## A3 (20 min): check by grid

**The idea** (*slide "When there is no formula: a grid"*). Instead of using the formula, compute the posterior numerically in four steps:

1. take many values of p between 0 and 1 (a *grid*);
2. at each value, compute likelihood × prior;
3. divide by the sum, so that the weights add up to 1: they then behave like probabilities;
4. posterior mean = sum(p × weight); P(p > 0.5 | y) = sum of the weights where p > 0.5.

**Tasks.**

1. Run the code cell. Compare the grid mean and grid probability with your exact answers from A1.
2. Why do we divide by the sum in step 3? What would the "mean" be without it?
3. Change `length.out = 2000` to `length.out = 5` and run again. What happens to the answers, and why?

In [ ]:
p_grid <- seq(0.0005, 0.9995, length.out = 2000)   # step 1: grid of p values
weight <- dbinom(y, n, p_grid) * dbeta(p_grid, 1, 1)  # step 2: likelihood x prior
weight <- weight / sum(weight)                         # step 3: weights add up to 1

c(grid_mean = sum(p_grid * weight),                    # step 4
  grid_prob = sum(weight[p_grid > 0.5]))
# Add your exact answers from A1 for comparison.

**Your answer:** write a few sentences here.

## Exit response

Write 80–120 words: What does the posterior interval describe, and why does changing the prior change the answer?

**Your exit response:**

## Your group's question set

Question writers: write **three** questions about this block, each with your own worked solution:

1. a **concept** question (what does a result mean?),
2. a **computation** question (a number someone can check in R),
3. a **pitfall** question (a mistake that is easy to make).

The checker reviews the solutions. The reporter notes what your group found hardest, to ask in the plenary.

**Concept question:**

**Our solution:**

**Computation question:**

**Our solution:**

**Pitfall question:**

**Our solution:**

**Hardest open question (reporter):**

---
**Everyone back in the room at 11:10 for Peer review A.** Save this notebook (Ctrl/Cmd + S). After lunch, open `day-2-exercises-afternoon.ipynb`.